# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pr120107/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:

import os
import subprocess
import sys

REPO_URL = "https://github.com/pr120107/flyrank-ml-internship.git"
REPO_DIR = "/content/flyrank-ml-internship"

# Clone the repo if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository
os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print("Repo contents:", os.listdir()[:10])

Working directory: /content/flyrank-ml-internship
Repo contents: ['SETUP.md', 'README.md', '.git', 'outputs', 'scripts', '.github', 'DATA_USE.md', 'data', 'work', 'requirements.txt']


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Health Score feature importance

The paper reports a Random Forest feature-importance analysis for predicting the FlyRank Health Score. The four largest features are Average Position (43), Impressions (32), Scroll Depth (15), and CTR (8).

My methodology question is: how much of this feature importance is genuinely learned by the model, versus being expected because the Health Score is itself constructed from these same four inputs? The paper defines the Health Score as Impressions (30 points) + Position (30 points) + CTR (20 points) + Scroll Depth (20 points).

This means the result should be interpreted as descriptive evidence that the model can reconstruct the composite score, rather than evidence that these features independently cause content to be healthy. The paper itself notes that the target is partly constructed from these inputs and that the importance is descriptive rather than causal.

### Finding 2 — Logistic Regression holdout accuracy

The paper reports 71% holdout accuracy for a Logistic Regression model describing which sampled features separate growing from declining pages.

My methodology question is: what exactly does this 71% holdout accuracy represent? The paper identifies the model as using an 80/20 split, but the reported result does not provide enough detail about the split boundary to determine whether the test set represents genuinely unseen brands or simply unseen rows from brands that also appear in training.

Therefore, the result is useful as an exploratory classification result, but I would be cautious about interpreting 71% accuracy as evidence that the model will generalize to new brands or future data without knowing more about the validation design.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
# ============================================
# Section 2 — Honest grouped-by-client split
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Load the same Week-5 dataset
df_model = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Recreate the same target
df_model["is_declining_label"] = (
    df_model["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Recreate the exact Week-5 feature set
y = df_model["is_declining_label"]

exclude_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

X = df_model.drop(columns=exclude_cols)

# --------------------------------------------
# Honest split: keep entire clients together
# --------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df_model["client_id"])
)

X_train_honest = X.iloc[train_idx]
X_test_honest = X.iloc[test_idx]

y_train_honest = y.iloc[train_idx]
y_test_honest = y.iloc[test_idx]

print("Honest grouped split")
print("--------------------")
print("Train size:", X_train_honest.shape)
print("Test size:", X_test_honest.shape)

print("\nNumber of clients in train:",
      df_model.iloc[train_idx]["client_id"].nunique())

print("Number of clients in test:",
      df_model.iloc[test_idx]["client_id"].nunique())

# Verify that no client appears in both sets
train_clients = set(df_model.iloc[train_idx]["client_id"])
test_clients = set(df_model.iloc[test_idx]["client_id"])

print("Clients appearing in both sets:",
      len(train_clients.intersection(test_clients)))

# --------------------------------------------
# Same preprocessing as Week 5
# --------------------------------------------

categorical_features = X_train_honest.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X_train_honest.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor_honest = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

# Same Decision Tree as Week 5
model_honest = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

pipeline_honest = Pipeline(
    steps=[
        ("preprocessor", preprocessor_honest),
        ("model", model_honest)
    ]
)

# Train
pipeline_honest.fit(
    X_train_honest,
    y_train_honest
)

# Predict
y_pred_honest = pipeline_honest.predict(X_test_honest)

# Evaluate
honest_accuracy = accuracy_score(
    y_test_honest,
    y_pred_honest
)

honest_precision = precision_score(
    y_test_honest,
    y_pred_honest,
    zero_division=0
)

honest_recall = recall_score(
    y_test_honest,
    y_pred_honest,
    zero_division=0
)

honest_f1 = f1_score(
    y_test_honest,
    y_pred_honest,
    zero_division=0
)

print("\nHonest-split results")
print("--------------------")
print(f"Accuracy:  {honest_accuracy:.4f}")
print(f"Precision: {honest_precision:.4f}")
print(f"Recall:    {honest_recall:.4f}")
print(f"F1:        {honest_f1:.4f}")

print("\nBefore vs After")
print("--------------------")
print("Week-5 random-split F1: 0.7790")
print(f"Week-6 grouped-split F1: {honest_f1:.4f}")
print(f"Change: {(honest_f1 - 0.7790):+.4f}")

Honest grouped split
--------------------
Train size: (23837, 40)
Test size: (6163, 40)

Number of clients in train: 25
Number of clients in test: 7
Clients appearing in both sets: 0

Honest-split results
--------------------
Accuracy:  0.6565
Precision: 0.6499
Recall:    0.7104
F1:        0.6788

Before vs After
--------------------
Week-5 random-split F1: 0.7790
Week-6 grouped-split F1: 0.6788
Change: -0.1002


### Interpretation

The Week-5 model achieved an F1 score of 0.7790 under a random 80/20 split. When evaluated under a grouped split by `client_id`, with no clients appearing in both training and test sets, F1 decreased to 0.6788.

This is a drop of 0.1002 F1 points. The grouped split is a more demanding test of generalization to unseen clients because the model cannot learn from other pages belonging to the same client during training.

Therefore, the Week-5 random-split result appears optimistic relative to performance on unseen clients. I would not interpret the 0.7790 F1 score as evidence of generalization to new clients without this additional validation.

The grouped result does not prove that the random split is invalid or that client-level leakage caused the difference. It shows that validation design materially affects the measured performance and that the model's generalization should be reported more cautiously.

## 3. Leakage Audit

The goal of this audit is to check whether the final Week-5 feature set contains variables that would not be legitimately available at the time a page is being screened for possible decline.

The target `is_declining_label` is derived from `trend_direction`, so `trend_direction` and `trend_pct` were excluded from the model features in Week 5.

I will also check for features that are very close to the target definition or could indirectly encode the outcome. This is important because strong predictive performance can be misleading if a feature contains information that would only become known after the outcome has already occurred.

The audit focuses on:
1. Direct target leakage.
2. Features derived from the target or its defining fields.
3. Features whose timing could make them unavailable at decision time.
4. Whether the strongest Week-5 features should be interpreted cautiously.

In [5]:
# Section 3: Leakage audit of the final Week-5 feature set

# Reconstruct the Week-5 feature list from the same dataset definition.
df_audit = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df_audit["is_declining_label"] = (
    df_audit["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

target = "is_declining_label"

exclude_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

feature_cols = [
    col for col in df_audit.columns
    if col not in exclude_cols
]

print("Leakage audit")
print("--------------------")
print("Target:", target)
print("Number of final model features:", len(feature_cols))
print("\nExcluded columns:")
for col in exclude_cols:
    print("-", col)

print("\nPotentially sensitive feature groups:")
for col in feature_cols:
    col_lower = col.lower()
    if any(term in col_lower for term in [
        "trend", "declin", "change", "growth", "loss", "drop"
    ]):
        print("-", col)

Leakage audit
--------------------
Target: is_declining_label
Number of final model features: 40

Excluded columns:
- content_id
- client_id
- trend_direction
- trend_pct
- is_declining_label

Potentially sensitive feature groups:


### Leakage audit findings

No direct target field is included in the final Week-5 feature set.

The target is constructed from `trend_direction`, and both `trend_direction` and `trend_pct` are explicitly excluded from the model. This prevents the model from directly receiving the variable used to define the declining label or the numerical trend measure from which that direction is derived.

The remaining features still require a timing check. In particular, metrics such as impressions, clicks, average position, engagement rate, and other performance measures may be legitimate predictors if they are known before the screening decision. However, their availability must be defined relative to the intended decision point.

The Week-5 feature-importance result should therefore be treated as model-specific evidence rather than proof that these variables cause decline. The three strongest transformed features were `impressions_prev_30d`, `impressions_last_30d`, and `content_age_days`, which together accounted for approximately 96.6% of the tree's total feature importance.

This audit does not establish that these features are leaked. It establishes that the direct target variables were excluded and that the remaining features need to be interpreted according to their timing and relationship to the decision point.

In [6]:
# Check the strongest Week-5 features against the raw dataset.

strong_features = [
    "impressions_prev_30d",
    "impressions_last_30d",
    "content_age_days",
    "avg_position",
    "days_since_last_update"
]

print("Timing / availability check")
print("--------------------")

for feature in strong_features:
    if feature in df_audit.columns:
        print(f"{feature}: present in dataset")
    else:
        print(f"{feature}: NOT FOUND")

Timing / availability check
--------------------
impressions_prev_30d: present in dataset
impressions_last_30d: present in dataset
content_age_days: present in dataset
avg_position: present in dataset
days_since_last_update: present in dataset


### Audit conclusion

The final feature set does not contain the target-defining variables `trend_direction` or `trend_pct`, so there is no direct target leakage from those fields.

However, leakage is not only a question of whether a target column is present. The timing of each feature also matters. A feature can be predictive without being useful for a real decision if it is measured after the outcome or after the intended intervention point.

For this reason, the Week-5 model should be described as a **decision-support screening model based on observed page-level signals**, not as evidence of causal drivers of decline.

The grouped validation result strengthens this caution: F1 decreased from 0.7790 under the random split to 0.6788 when evaluated on completely unseen clients.

## 4. Claim Rewrite

### Original claim

The model identifies the key factors that cause content to decline.

### Safer rewritten claim

The model found that observed page-level signals, particularly recent and previous 30-day impressions and content age, were strongly associated with the `is_declining_label` in this dataset. Under a grouped client-level validation split, the model achieved an F1 score of 0.6788, compared with 0.7790 under the Week-5 random split.

These results are directional and model-specific. They indicate that the observed signals may be useful for prioritizing pages for review, but they do not establish that these features cause content decline or that the model will generalize equally well to unseen clients.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.